In [1]:
import pandas as pd
import re
import os
import nltk
import html
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer



In [2]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
TRAIN_PATH   = '../data/raw/train.csv'
TEST_PATH    = '../data/raw/test.csv'
OUTPUT_DIR   = '../data/processed'
SAMPLE_SIZE  = None

os.makedirs(OUTPUT_DIR, exist_ok=True)


## Load Data

In [4]:
df = pd.read_csv(
    TRAIN_PATH,
    header=0,
    names=['label', 'text'],
    nrows=SAMPLE_SIZE
)

print(f'Shape   : {df.shape}')
df.head()


Shape   : (650000, 2)


,label,text
0,4,dr. goldberg offers everything i look for in a...
1,1,"Unfortunately, the frustration of being Dr. Go..."
2,3,Been going to Dr. Goldberg for over 10 years. ...
3,3,Got a letter in the mail last week that said D...
4,0,I don't know what Dr. Goldberg was like before...


## Data Cleaning

In [5]:
# Missing values
df.isnull().sum()

label    0
text     0
dtype: int64

In [6]:
# Duplicate rows
df.duplicated().sum()

0

In [7]:
df.dropna(subset=['text'], inplace=True)
df.drop_duplicates(subset=['text'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [8]:
#    Convert 5-class label to binary.
# Positive (1) = labels 3, 4  (4 & 5 stars)
# Negative (0) = labels 0, 1  (1 & 2 stars)
# Neutral  (2) = label  2     (3 stars)
def convert_to_binary_label(label: int) -> int:
    return 1 if label >= 3 else 0


## Preprocessing Pipeline

In [9]:
# Lowercase
def to_lowercase(text: str) -> str:
    return text.lower()

In [10]:
# Remove URLs
def remove_urls(text: str) -> str:
    return re.sub(r'https?://\S+|www\.\S+', '', text)

In [11]:
# Remove HTML tags
def remove_html_tags(text: str) -> str:
    text = html.unescape(text)
    return re.sub(r'<.*?>', ' ', text)

In [12]:
# Remove all punctuation characters
def remove_newlines_and_punctuation(text: str) -> str:
    text = re.sub(r'\\n|\n', ' ', text)
    return re.sub(r'[^\w\s]', ' ', text)

In [13]:
# removes digits & symbols
def remove_special_characters(text: str) -> str:
    return re.sub(r'[^a-z\s]', '', text)

In [14]:
# Remove extra whitespace
def remove_extra_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

In [15]:
# Split text into individual word tokens
def tokenize(text: str) -> list:
    return word_tokenize(text)

In [16]:
# Remove stopwords but KEEP negation words
base_stopwords = set(stopwords.words('english'))

negations = {
    'no', 'not', 'nor', 'none', 'neither', 'never', 'nobody', 
    'dont', 'doesnt', 'didnt', 'isnt', 'arent', 'wasnt', 'werent', 
    'hasnt', 'havent', 'hadnt', 'wont', 'wouldnt', 'cant', 'couldnt', 
    'shouldnt', 'mightnt', 'mustnt'
}

STOP_WORDS = base_stopwords - negations

def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in STOP_WORDS]


In [17]:
stemmer = SnowballStemmer("english")

def stem_tokens(tokens: list) -> list:
    return [stemmer.stem(t) for t in tokens]

In [18]:
def preprocess_text(text: str) -> str:
    text = to_lowercase(text)
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = remove_newlines_and_punctuation(text)
    text = remove_special_characters(text)
    text = remove_extra_spaces(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = stem_tokens(tokens)
    return ' '.join(tokens)


In [19]:
# Drop neutral reviews
original_len = len(df)
df = df[df['label'] != 2].reset_index(drop=True)
print(f'Removed neutral reviews : {original_len - len(df):,}')
print(f'Remaining rows : {len(df):,}')

Removed neutral reviews : 130,000
Remaining rows : 520,000


In [20]:
df['binary_label'] = df['label'].apply(convert_to_binary_label)
df['cleaned_text'] = df['text'].apply(preprocess_text)


In [21]:
# Word count statistics
df['original_word_count'] = df['text'].apply(lambda x: len(x.split()))
df['cleaned_word_count']  = df['cleaned_text'].apply(lambda x: len(x.split()))

print('WORD COUNT STATISTICS')
print('\nOriginal text:')
print(df['original_word_count'].describe().round(1))
print('\nCleaned text:')
print(df['cleaned_word_count'].describe().round(1))

reduction = (1 - df['cleaned_word_count'].mean() / df['original_word_count'].mean()) * 100
print(f'\nAverage word reduction: {reduction:.1f}%')

WORD COUNT STATISTICS

Original text:
count    520000.0
mean        133.0
std         122.5
min           1.0
25%          51.0
50%          97.0
75%         173.0
max        1052.0
Name: original_word_count, dtype: float64

Cleaned text:
count    520000.0
mean         67.2
std          60.9
min           0.0
25%          27.0
50%          49.0
75%          87.0
max         912.0
Name: cleaned_word_count, dtype: float64

Average word reduction: 49.5%


In [22]:
# Filtering short reviews
original_len = len(df)

df = df[df['cleaned_text'].apply(lambda x: len(str(x).split()) >= 2)].reset_index(drop=True)

print(f'Rows removed: {original_len - len(df)}')
print(f'Remaining rows: {len(df):,}')

Rows removed: 459
Remaining rows: 519,541


In [23]:
# Before vs After preview
for i, row in df.sample(3, random_state=42).iterrows():
    print(f"\n[binary_label: {row['binary_label']}]")
    print(f"  Original : {row['text'][:200]}")
    print(f"  Cleaned  : {row['cleaned_text'][:200]}")


[binary_label: 0]
  Original : This may have been a nice experience if it hadn't been for the grumpy waiter. He was definitely in a foul mood and seemed bothered we were there. When our food was delivered he did not place our plate
  Cleaned  : may nice experi grumpi waiter definit foul mood seem bother food deliv not place plate front us drop slam startl never came back check us came back give us bill food ok never back

[binary_label: 1]
  Original : I have never been to a haunted house like this. Going through it is just like stepping into an Eli Roth movie set. So disturbing. I loved it!
  Cleaned  : never haunt hous like go like step eli roth movi set disturb love

[binary_label: 1]
  Original : What a great record store!\n\nEvery city has a record store like this.  SF has Amoeba, NY has Bleecker Music and so on.  Jerry's has Pittsburg covered.\n\nIt's just outside of the city in a second flo
  Cleaned  : great record store everi citi record store like sf amoeba ny bleecker music

## Save Train Set

In [24]:
df_train_final = df[['binary_label', 'text', 'cleaned_text']].copy()

print(f'Shape   : {df_train_final.shape}')
df_train_final.head()

Shape   : (519541, 3)


,binary_label,text,cleaned_text
0,1,dr. goldberg offers everything i look for in a...,dr goldberg offer everyth look general practit...
1,0,"Unfortunately, the frustration of being Dr. Go...",unfortun frustrat dr goldberg patient repeat e...
2,1,Been going to Dr. Goldberg for over 10 years. ...,go dr goldberg year think one st patient start...
3,1,Got a letter in the mail last week that said D...,got letter mail last week said dr goldberg mov...
4,0,I don't know what Dr. Goldberg was like before...,know dr goldberg like move arizona let tell st...


In [25]:
train_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_train.csv')
df_train_final.to_csv(train_output_path, index=False)


In [26]:
# Train label distribution
print('Label distribution (Train set):')
dist = df_train_final['binary_label'].value_counts().sort_index()
total = len(df_train_final)

for label, count in dist.items():
    sentiment = 'Positive' if label == 1 else 'Negative'
    print(f'  {sentiment} ({label}) : {count:>7,} rows ({count/total*100:5.1f}%)')

print(f'\n  Total : {total:>7,} rows')

Label distribution (Train set):
  Negative (0) : 259,709 rows ( 50.0%)
  Positive (1) : 259,832 rows ( 50.0%)

  Total : 519,541 rows


## Apply Pipeline — Test Set

In [27]:
df_test = pd.read_csv(TEST_PATH, header=0, names=['label', 'text'])

df_test = df_test[df_test['label'] != 2].reset_index(drop=True)

df_test['binary_label'] = df_test['label'].apply(convert_to_binary_label)
df_test['cleaned_text'] = df_test['text'].apply(preprocess_text)

# Drop rows where cleaned_text has less than 2 words
df_test = df_test[df_test['cleaned_text'].apply(lambda x: len(str(x).split()) >= 2)].reset_index(drop=True)


df_test_final = df_test[['binary_label', 'text', 'cleaned_text']].copy()

print(f'Test set ready : {len(df_test_final):,} rows')
df_test_final.head()

Test set ready : 39,969 rows


,binary_label,text,cleaned_text
0,0,I got 'new' tires from them and within two wee...,got new tire within two week got flat took car...
1,0,Don't waste your time. We had two different p...,wast time two differ peopl come hous give us e...
2,0,All I can say is the worst! We were the only 2...,say worst peopl place lunch place freez load k...
3,0,I have been to this restaurant twice and was d...,restaur twice disappoint time go back first ti...
4,0,Food was NOT GOOD at all! My husband & I ate h...,food not good husband ate coupl week ago first...


## Save Test Set

In [28]:
test_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_test.csv')
df_test_final.to_csv(test_output_path, index=False)

In [29]:
# Test label distribution
print('Label distribution (Test set):')
dist = df_test_final['binary_label'].value_counts().sort_index()
total = len(df_test_final)

for label, count in dist.items():
    sentiment = 'Positive' if label == 1 else 'Negative'
    print(f'  {sentiment} ({label}) : {count:>7,} rows ({count/total*100:5.1f}%)')

print(f'\n  Total : {total:>7,} rows')

Label distribution (Test set):
  Negative (0) :  19,980 rows ( 50.0%)
  Positive (1) :  19,989 rows ( 50.0%)

  Total :  39,969 rows
